# UMUD Challenge: 4-Model Pipeline (UNet++ & GPU XGBoost)

This notebook implements the complete 6-stage pipeline for automated muscle architecture estimation:
- **Stage 0:** Robust dataset location auto-discovery & stratified 4-fold split.
- **Stage 1:** Reference geometry (DL_Track_US logic) & pseudo-labeling with robust outlier handling.
- **Stage 2:** **Model 1 & 2 (UNet++)** with timm-efficientnet-b2 encoder and mixed precision (FP16).
- **Stage 3:** Feature engineering from out-of-fold and test probability maps.
- **Stage 4:** **Model 3 & 4 (GPU XGBoost)** predicting residual variations from resolution-stratified medians on clean non-NaN targets.
- **Stage 5 & 6:** Shrinkage calibration, percentile bounding, and submission generation.

### Fixes applied:
- Updated `torch.amp` API (non-deprecated `torch.amp.autocast`/`torch.amp.GradScaler`)
- FL consistency: FL = MT / sin(PA) enforced everywhere
- `fallback_fl` computed from actual geometry, not hardcoded
- Resolution-adaptive Gaussian sigma for post-processing
- Component detection uses `np.bincount` for speed
- Retry with lower threshold before falling back to global medians
- Best-epoch checkpoint saving (lowest validation Dice loss)
- Robust file extension filtering

In [ ]:
# Install dependencies in Kaggle environment
!pip install -q segmentation-models-pytorch albumentations

In [ ]:
import os
import glob
import math
import random
import warnings
import numpy as np
import pandas as pd
import cv2
from PIL import Image
from tqdm.auto import tqdm
from scipy.ndimage import label, center_of_mass

import torch
import torch.nn as nn
# Use non-deprecated torch.amp API
from torch.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import StratifiedKFold
import xgboost as xgb
import segmentation_models_pytorch as smp

warnings.filterwarnings('ignore')

print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device Name:', torch.cuda.get_device_name(0))
    print('GPU Count:', torch.cuda.device_count())

In [ ]:
def locate_dataset():
    candidates = [
        '/kaggle/input/competitions/umud-challenge-muscle-architecture-in-ultrasound-data',
        '/kaggle/input/umud-challenge-muscle-architecture-in-ultrasound-data',
        './umud-challenge-muscle-architecture-in-ultrasound-data',
    ]
    
    if os.path.exists('/kaggle/input'):
        for root, dirs, _ in os.walk('/kaggle/input'):
            if 'apo_imgs_v1' in dirs or any('apo_imgs' in d for d in dirs):
                candidates.insert(0, root)
                break
                
    for c in candidates:
        if os.path.exists(c):
            contents = os.listdir(c)
            if any('apo_imgs' in x for x in contents):
                return c
    return candidates[0]

found_dir = locate_dataset()
print(f'Discovered dataset base dir: {found_dir}')

def find_leaf_dir(parent_dir, sub_name, nested_hint):
    target = os.path.join(parent_dir, sub_name)
    if os.path.exists(target):
        sub_items = [d for d in os.listdir(target) if os.path.isdir(os.path.join(target, d))]
        if len(sub_items) > 0:
            for d in sub_items:
                if nested_hint in d:
                    return os.path.join(target, d)
            return os.path.join(target, sub_items[0])
        return target
    return target

class CFG:
    seed = 42
    n_folds = 4
    img_size = (384, 576)  # Height x Width (2:3 aspect ratio)
    batch_size = 16        # Fits smoothly on Kaggle T4 / P100 16GB VRAM
    num_workers = 4
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    use_amp = True         # FP16 mixed precision
    
    # UNet++ Architecture
    encoder_name = 'timm-efficientnet-b2'
    encoder_weights = 'imagenet'
    lr = 5e-4
    epochs = 10
    
    base_dir = found_dir
    
    apo_img_dir = find_leaf_dir(base_dir, 'apo_imgs_v1', 'apo_images')
    apo_mask_dir = find_leaf_dir(base_dir, 'apo_masks_v1', 'apo_masks')
    fasc_img_dir = find_leaf_dir(base_dir, 'fasc_imgs_v1', 'fasc_images')
    fasc_mask_dir = find_leaf_dir(base_dir, 'fasc_masks_v1', 'fasc_masks')
    test_img_dir = find_leaf_dir(base_dir, 'test_images_v2', 'test_set')
    
    # Robust output directory: works on Kaggle, Colab, and local
    output_dir = '/kaggle/working' if os.path.exists('/kaggle/working') else './pipeline_outputs'

os.makedirs(CFG.output_dir, exist_ok=True)

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = True

seed_everything(CFG.seed)
print(f'Using base directory: {CFG.base_dir}')
print(f'Apo images dir: {CFG.apo_img_dir} (exists: {os.path.exists(CFG.apo_img_dir)})')
print(f'Fasc images dir: {CFG.fasc_img_dir} (exists: {os.path.exists(CFG.fasc_img_dir)})')
print(f'Test images dir: {CFG.test_img_dir} (exists: {os.path.exists(CFG.test_img_dir)})')
print(f'Output directory: {CFG.output_dir}')

In [ ]:
# Supported image extensions for safe file listing
VALID_IMG_EXTS = ('.tif', '.tiff', '.png', '.jpg', '.jpeg', '.bmp')

def get_image_dict(directory):
    """Scan directory recursively for image files, filtering by extension."""
    files = {}
    for ext in ['*.tif', '*.tiff', '*.png', '*.jpg', '*.jpeg', '*.bmp']:
        for f in glob.glob(os.path.join(directory, '**', ext), recursive=True):
            basename = os.path.basename(f)
            # Skip hidden files, macOS artifacts, and non-image files
            if basename.startswith('.') or basename.lower().startswith('thumbs'):
                continue
            files[basename] = f
    return files

def build_dataset_index():
    print('--- [Stage 0] Auditing Data & Building Stratified Index ---')
    apo_files = get_image_dict(CFG.apo_img_dir)
    fasc_files = get_image_dict(CFG.fasc_img_dir)
    test_files = get_image_dict(CFG.test_img_dir)
    
    print(f'Discovered files -> Apo: {len(apo_files)}, Fasc: {len(fasc_files)}, Test: {len(test_files)}')
    
    if len(apo_files) == 0 and len(fasc_files) == 0:
        raise RuntimeError(f'No training images found in {CFG.base_dir}. Please verify that the Kaggle competition dataset is attached!')
    
    # Cross-check: match images to masks by stripping extensions and re-matching
    apo_mask_files = get_image_dict(CFG.apo_mask_dir)
    fasc_mask_files = get_image_dict(CFG.fasc_mask_dir)
    
    # Build stem-to-filename maps for flexible matching (handles extension mismatches)
    def stem_map(file_dict):
        return {os.path.splitext(k)[0]: k for k in file_dict}
    
    apo_img_stems = stem_map(apo_files)
    apo_mask_stems = stem_map(apo_mask_files)
    fasc_img_stems = stem_map(fasc_files)
    fasc_mask_stems = stem_map(fasc_mask_files)
    
    # Warn about extension mismatches
    matched_apo = set(apo_img_stems.keys()) & set(apo_mask_stems.keys())
    matched_fasc = set(fasc_img_stems.keys()) & set(fasc_mask_stems.keys())
    unmatched_apo = set(apo_img_stems.keys()) - set(apo_mask_stems.keys())
    unmatched_fasc = set(fasc_img_stems.keys()) - set(fasc_mask_stems.keys())
    
    if unmatched_apo:
        print(f'WARNING: {len(unmatched_apo)} apo images have no matching mask (first 5: {list(unmatched_apo)[:5]})')
    if unmatched_fasc:
        print(f'WARNING: {len(unmatched_fasc)} fasc images have no matching mask (first 5: {list(unmatched_fasc)[:5]})')
    
    records = []
    all_train_ids = set(apo_files.keys()).union(set(fasc_files.keys()))
    
    for img_id in all_train_ids:
        has_apo = img_id in apo_files
        has_fasc = img_id in fasc_files
        path = apo_files[img_id] if has_apo else fasc_files[img_id]
        
        # Also check if the mask exists for this exact filename
        has_apo_mask = img_id in apo_mask_files
        has_fasc_mask = img_id in fasc_mask_files
        
        with Image.open(path) as img:
            w, h = img.size
            
        res_cluster = f'{w}x{h}'
        split = 'dual' if (has_apo and has_fasc) else ('fasc_only' if has_fasc else 'apo_only')
        scale_val = 0.0643  # default ~0.064 mm/pixel
        
        records.append({
            'image_id': img_id,
            'split': split,
            'width': w,
            'height': h,
            'resolution_cluster': res_cluster,
            'scale_val': scale_val,
            'has_apo': has_apo and has_apo_mask,  # Only True if both image and mask exist
            'has_fasc': has_fasc and has_fasc_mask,
            'is_test': False
        })
        
    for img_id, path in test_files.items():
        with Image.open(path) as img:
            w, h = img.size
        records.append({
            'image_id': img_id,
            'split': 'test',
            'width': w,
            'height': h,
            'resolution_cluster': f'{w}x{h}',
            'scale_val': 0.0643,
            'has_apo': False,
            'has_fasc': False,
            'is_test': True
        })
        
    df = pd.DataFrame(records)
    df['fold'] = -1
    skf = StratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
    
    dual_mask = df['split'] == 'dual'
    if dual_mask.sum() >= CFG.n_folds:
        for fold, (_, val_idx) in enumerate(skf.split(df[dual_mask], df[dual_mask]['resolution_cluster'])):
            df.loc[df[dual_mask].iloc[val_idx].index, 'fold'] = fold
    else:
        print(f'WARNING: Only {dual_mask.sum()} dual images, assigning all to fold 0')
        df.loc[dual_mask, 'fold'] = 0
        
    fasc_mask = df['split'] == 'fasc_only'
    if fasc_mask.sum() >= CFG.n_folds:
        for fold, (_, val_idx) in enumerate(skf.split(df[fasc_mask], df[fasc_mask]['resolution_cluster'])):
            df.loc[df[fasc_mask].iloc[val_idx].index, 'fold'] = fold

    df.to_csv(os.path.join(CFG.output_dir, 'dataset_folds.csv'), index=False)
    print(f'Dual Images: {dual_mask.sum()}, Fasc-Only Images: {fasc_mask.sum()}, Test Images: {df["is_test"].sum()}')
    return df

df_meta = build_dataset_index()

In [ ]:
def read_image_safe(path, mode='rgb'):
    """Fast and clean image reading with PIL to avoid OpenCV libtiff tag warnings."""
    with Image.open(path) as img:
        if mode == 'rgb':
            return np.array(img.convert('RGB'))
        else:
            return np.array(img.convert('L'))

def compute_muscle_geometry(apo_mask, fasc_mask, mm_per_px=0.0643):
    """Compute MT, PA, FL from aponeurosis and fascicle masks.
    
    Uses np.bincount for fast component sizing instead of repeated (lbl == i) scans.
    Ensures FL = MT / sin(PA) geometric consistency.
    """
    H, W = apo_mask.shape
    apo_binary = (apo_mask > 127).astype(np.uint8)
    fasc_binary = (fasc_mask > 127).astype(np.uint8)
    
    # 1. Muscle Thickness (MT) - find superficial and deep aponeurosis
    labeled_apo, num_comp = label(apo_binary)
    
    if num_comp >= 2:
        # Use bincount for fast component size computation
        comp_sizes = np.bincount(labeled_apo.ravel())
        # Get centroids for all components
        comps = []
        for i in range(1, num_comp + 1):
            if comp_sizes[i] < 10:  # Skip tiny fragments
                continue
            cy, _ = center_of_mass(labeled_apo == i)
            comps.append((cy, i, comp_sizes[i]))
        
        if len(comps) >= 2:
            # Sort by y-position (top to bottom)
            comps.sort(key=lambda x: x[0])
            # Take the topmost and bottommost as superficial/deep
            sup_idx = comps[0][1]
            deep_idx = comps[-1][1]
            sup_mask_arr = (labeled_apo == sup_idx)
            deep_mask_arr = (labeled_apo == deep_idx)
        else:
            # Only 1 significant component, split by midline
            sup_mask_arr = np.zeros_like(apo_binary, dtype=bool)
            deep_mask_arr = np.zeros_like(apo_binary, dtype=bool)
            sup_mask_arr[:H // 2, :] = apo_binary[:H // 2, :] > 0
            deep_mask_arr[H // 2:, :] = apo_binary[H // 2:, :] > 0
    else:
        sup_mask_arr = np.zeros_like(apo_binary, dtype=bool)
        deep_mask_arr = np.zeros_like(apo_binary, dtype=bool)
        sup_mask_arr[:H // 2, :] = apo_binary[:H // 2, :] > 0
        deep_mask_arr[H // 2:, :] = apo_binary[H // 2:, :] > 0

    sup_coords = np.argwhere(sup_mask_arr)
    deep_coords = np.argwhere(deep_mask_arr)
    
    if len(sup_coords) < 10 or len(deep_coords) < 10:
        return {'mt_mm': np.nan, 'pa_deg': np.nan, 'fl_mm': np.nan, 'reliability': 0.0}
    
    mean_sup_y = np.mean(sup_coords[:, 0])
    mean_deep_y = np.mean(deep_coords[:, 0])
    mt_pixels = abs(mean_deep_y - mean_sup_y)
    mt_mm = mt_pixels * mm_per_px
    
    # 2. Pennation Angle (PA) & Fascicle Length (FL)
    fasc_pts = np.argwhere(fasc_binary > 0)
    if len(fasc_pts) < 15:
        return {'mt_mm': mt_mm, 'pa_deg': np.nan, 'fl_mm': np.nan, 'reliability': 0.3}
    
    y_coords, x_coords = fasc_pts[:, 0], fasc_pts[:, 1]
    slope, _ = np.polyfit(x_coords, y_coords, deg=1)
    
    pa_deg = abs(np.degrees(np.arctan(slope)))
    pa_rad = np.radians(max(pa_deg, 3.0))  # Floor at 3 degrees to avoid division by near-zero
    
    # FL = MT / sin(PA) — enforced geometric consistency
    fl_mm = mt_mm / np.sin(pa_rad)
    
    is_valid = (5.0 <= mt_mm <= 60.0) and (2.0 <= pa_deg <= 45.0) and (15.0 <= fl_mm <= 250.0)
    reliability = 1.0 if is_valid else 0.5
    
    return {'mt_mm': mt_mm, 'pa_deg': pa_deg, 'fl_mm': fl_mm, 'reliability': reliability}

def generate_reference_pseudo_labels(df):
    print('--- [Stage 1] Computing Reference Pseudo-Labels ---')
    pseudo_records = []
    fallback_pa, fallback_mt, fallback_fl = [], [], []  # Collect valid values for fallback
    
    for _, row in tqdm(df[df['split'] == 'dual'].iterrows(), total=(df['split'] == 'dual').sum()):
        apo_p = os.path.join(CFG.apo_mask_dir, row['image_id'])
        fasc_p = os.path.join(CFG.fasc_mask_dir, row['image_id'])
        
        if not os.path.exists(apo_p) or not os.path.exists(fasc_p):
            pseudo_records.append({
                'image_id': row['image_id'],
                'resolution_cluster': row['resolution_cluster'],
                'mt_mm': np.nan, 'pa_deg': np.nan, 'fl_mm': np.nan, 'reliability': 0.0
            })
            continue
        
        apo_m = read_image_safe(apo_p, mode='gray')
        fasc_m = read_image_safe(fasc_p, mode='gray')
        
        if apo_m.shape != (row['height'], row['width']):
            apo_m = cv2.resize(apo_m, (row['width'], row['height']), interpolation=cv2.INTER_NEAREST)
        if fasc_m.shape != (row['height'], row['width']):
            fasc_m = cv2.resize(fasc_m, (row['width'], row['height']), interpolation=cv2.INTER_NEAREST)
            
        geom = compute_muscle_geometry(apo_m, fasc_m, row['scale_val'])
        
        # Collect valid values for computing fallback medians
        if not np.isnan(geom['pa_deg']):
            fallback_pa.append(geom['pa_deg'])
        if not np.isnan(geom['mt_mm']):
            fallback_mt.append(geom['mt_mm'])
        if not np.isnan(geom['fl_mm']):
            fallback_fl.append(geom['fl_mm'])
        
        pseudo_records.append({
            'image_id': row['image_id'],
            'resolution_cluster': row['resolution_cluster'],
            **geom
        })
    
    # Compute fallback values from actual geometry, not hardcoded
    FALLBACK_PA = float(np.median(fallback_pa)) if fallback_pa else 14.87
    FALLBACK_MT = float(np.median(fallback_mt)) if fallback_mt else 19.37
    # FL must be geometrically consistent: FL = MT / sin(PA)
    FALLBACK_FL = float(np.median(fallback_fl)) if fallback_fl else (FALLBACK_MT / math.sin(math.radians(FALLBACK_PA)))
    print(f'Fallback values: PA={FALLBACK_PA:.2f}°, MT={FALLBACK_MT:.2f}mm, FL={FALLBACK_FL:.2f}mm')
    print(f'FL consistency check: MT/sin(PA) = {FALLBACK_MT / math.sin(math.radians(FALLBACK_PA)):.2f}mm')
        
    pseudo_df = pd.DataFrame(pseudo_records)
    group_medians = pseudo_df.groupby('resolution_cluster')[['pa_deg', 'fl_mm', 'mt_mm']].median()
    global_median = pseudo_df[['pa_deg', 'fl_mm', 'mt_mm']].median()
    
    # Store fallback values in global_median for downstream use
    global_median['fallback_pa'] = FALLBACK_PA
    global_median['fallback_mt'] = FALLBACK_MT
    global_median['fallback_fl'] = FALLBACK_FL
    
    pseudo_df.to_csv(os.path.join(CFG.output_dir, 'pseudo_labels.csv'), index=False)
    group_medians.to_csv(os.path.join(CFG.output_dir, 'group_medians.csv'))
    print('\nGroup Medians Table:\n', group_medians)
    return pseudo_df, group_medians, global_median

pseudo_df, group_medians, global_median = generate_reference_pseudo_labels(df_meta)

In [ ]:
class UltrasoundSegDataset(Dataset):
    def __init__(self, df, mode='apo', transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.mode = mode
        self.transform = transform
        self.is_test = is_test
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = row['image_id']
        
        if self.is_test:
            path = os.path.join(CFG.test_img_dir, img_id)
        else:
            path = os.path.join(CFG.apo_img_dir if row['has_apo'] else CFG.fasc_img_dir, img_id)
            
        image = read_image_safe(path, mode='rgb')
        
        if self.is_test:
            if self.transform:
                augmented = self.transform(image=image)
                image = augmented['image']
            return image, img_id
            
        mask_dir = CFG.apo_mask_dir if self.mode == 'apo' else CFG.fasc_mask_dir
        mask = read_image_safe(os.path.join(mask_dir, img_id), mode='gray')
        
        # Ensure mask matches image resolution
        if mask.shape[:2] != image.shape[:2]:
            mask = cv2.resize(mask, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_NEAREST)
            
        mask = (mask > 127).astype(np.float32)
        
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask'].unsqueeze(0)
            
        return image, mask, img_id

# ImageNet normalization applied (required for pretrained encoder)
train_transform = A.Compose([
    A.Resize(CFG.img_size[0], CFG.img_size[1]),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.08, rotate_limit=10, p=0.4),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
], is_check_shapes=False)

val_transform = A.Compose([
    A.Resize(CFG.img_size[0], CFG.img_size[1]),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
], is_check_shapes=False)

class DiceBCELoss(nn.Module):
    def __init__(self, pos_weight=1.0):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]))
        
    def forward(self, inputs, targets, smooth=1):
        bce_loss = self.bce(inputs, targets)
        inputs_sig = torch.sigmoid(inputs)
        intersection = (inputs_sig * targets).sum()
        dice_loss = 1 - (2. * intersection + smooth) / (inputs_sig.sum() + targets.sum() + smooth)
        return bce_loss + dice_loss

In [ ]:
def train_unetpp_model(df, mode='apo'):
    print(f'\n==========================================')
    print(f' Training UNet++ for {mode.upper()} on GPU ')
    print(f'==========================================')
    filter_col = 'has_apo' if mode == 'apo' else 'has_fasc'
    sub_df = df[df[filter_col] == True].copy()
    
    oof_predictions = {}
    # Use non-deprecated torch.amp.GradScaler
    scaler = GradScaler('cuda', enabled=CFG.use_amp)
    
    for fold in range(CFG.n_folds):
        ckpt_path = os.path.join(CFG.output_dir, f'unetpp_{mode}_fold{fold}.pth')
        val_df = sub_df[sub_df['fold'] == fold]
        val_ds = UltrasoundSegDataset(val_df, mode=mode, transform=val_transform)
        val_loader = DataLoader(
            val_ds, batch_size=CFG.batch_size, shuffle=False,
            num_workers=CFG.num_workers, pin_memory=True
        )
        
        model = smp.UnetPlusPlus(
            encoder_name=CFG.encoder_name,
            encoder_weights=CFG.encoder_weights,
            in_channels=3,
            classes=1
        ).to(CFG.device)
        
        if os.path.exists(ckpt_path):
            print(f'--- {mode.upper()} Fold {fold} Checkpoint Found! Loading cached weights... ---')
            model.load_state_dict(torch.load(ckpt_path, map_location=CFG.device, weights_only=True))
        else:
            print(f'\n--- {mode.upper()} Fold {fold} ---')
            train_df = sub_df[sub_df['fold'] != fold]
            train_ds = UltrasoundSegDataset(train_df, mode=mode, transform=train_transform)
            train_loader = DataLoader(
                train_ds, batch_size=CFG.batch_size, shuffle=True,
                num_workers=CFG.num_workers, pin_memory=True
            )
            
            pos_weight = 1.5 if mode == 'apo' else 4.0
            criterion = DiceBCELoss(pos_weight=pos_weight).to(CFG.device)
            optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=1e-4)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.epochs)
            
            best_val_loss = float('inf')
            
            for epoch in range(CFG.epochs):
                # --- Training ---
                model.train()
                train_loss = 0.0
                for imgs, masks, _ in train_loader:
                    imgs, masks = imgs.to(CFG.device, non_blocking=True), masks.to(CFG.device, non_blocking=True)
                    optimizer.zero_grad(set_to_none=True)
                    
                    # Use non-deprecated torch.amp.autocast
                    with autocast('cuda', enabled=CFG.use_amp):
                        preds = model(imgs)
                        loss = criterion(preds, masks)
                        
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                    train_loss += loss.item()
                
                scheduler.step()
                avg_train_loss = train_loss / len(train_loader)
                    
                # --- Validation (every epoch for best checkpoint) ---
                model.eval()
                val_loss = 0.0
                with torch.no_grad():
                    for imgs, masks, _ in val_loader:
                        imgs = imgs.to(CFG.device, non_blocking=True)
                        masks = masks.to(CFG.device, non_blocking=True)
                        with autocast('cuda', enabled=CFG.use_amp):
                            preds = model(imgs)
                            loss = criterion(preds, masks)
                        val_loss += loss.item()
                avg_val_loss = val_loss / max(len(val_loader), 1)
                
                # Save best checkpoint
                if avg_val_loss < best_val_loss:
                    best_val_loss = avg_val_loss
                    torch.save(model.state_dict(), ckpt_path)
                
                if (epoch + 1) % 2 == 0 or epoch == CFG.epochs - 1:
                    print(f'Epoch {epoch+1}/{CFG.epochs} - Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Best: {best_val_loss:.4f}')
            
            # Reload best checkpoint
            model.load_state_dict(torch.load(ckpt_path, map_location=CFG.device, weights_only=True))
        
        # Out-of-Fold inference (guaranteed float32 for downstream numpy processing)
        model.eval()
        with torch.no_grad():
            for imgs, _, img_ids in val_loader:
                imgs = imgs.to(CFG.device, non_blocking=True)
                with autocast('cuda', enabled=CFG.use_amp):
                    probs = torch.sigmoid(model(imgs)).float().cpu().numpy()[:, 0, :, :]
                for i_id, prob in zip(img_ids, probs):
                    oof_predictions[i_id] = np.ascontiguousarray(prob, dtype=np.float32)
                    
    return oof_predictions

oof_apo = train_unetpp_model(df_meta, mode='apo')
oof_fasc = train_unetpp_model(df_meta, mode='fasc')

In [ ]:
def extract_map_features(apo_prob, fasc_prob, scale_val, img_width=576):
    """Extract features from probability maps.
    
    Uses resolution-adaptive sigma for Gaussian smoothing.
    """
    features = {}
    
    apo_prob = np.ascontiguousarray(apo_prob, dtype=np.float32)
    fasc_prob = np.ascontiguousarray(fasc_prob, dtype=np.float32)
    
    # Resolution-adaptive sigma (scaled relative to 576px reference width)
    sigma_base = 12.0
    sigma = sigma_base * (img_width / 576.0)
    
    # Aponeurosis blob and thickness metrics
    for th in [0.3, 0.5, 0.7]:
        bin_apo = (apo_prob > th).astype(np.uint8)
        _, num_c = label(bin_apo)
        features[f'apo_num_blobs_th_{th}'] = float(num_c)
        features[f'apo_area_th_{th}'] = float(bin_apo.sum())
        
    geom = compute_muscle_geometry(apo_prob * 255.0, fasc_prob * 255.0, scale_val)
    features['geom_mt_mm'] = float(geom['mt_mm']) if not np.isnan(geom['mt_mm']) else -1.0
    features['geom_pa_deg'] = float(geom['pa_deg']) if not np.isnan(geom['pa_deg']) else -1.0
    features['geom_fl_mm'] = float(geom['fl_mm']) if not np.isnan(geom['fl_mm']) else -1.0
    features['reliability'] = float(geom['reliability'])
    
    # Fascicle dispersion and gradient stats
    features['fasc_mean_prob'] = float(np.mean(fasc_prob))
    features['fasc_max_prob'] = float(np.max(fasc_prob))
    features['fasc_area_03'] = float(np.sum(fasc_prob > 0.3))
    
    # Structure tensor / Sobel energy (guaranteed CV_32F)
    sobel_x = cv2.Sobel(fasc_prob, cv2.CV_32F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(fasc_prob, cv2.CV_32F, 0, 1, ksize=3)
    features['fasc_gradient_mag'] = float(np.mean(np.sqrt(sobel_x**2 + sobel_y**2)))
    features['scale_val'] = float(scale_val)
    features['img_width'] = float(img_width)
    
    return features

print('--- [Stage 3] Building OOF Feature Table ---')
feature_records = []
for img_id in tqdm(oof_apo.keys()):
    if img_id in oof_fasc:
        row_meta = df_meta[df_meta['image_id'] == img_id].iloc[0]
        f = extract_map_features(oof_apo[img_id], oof_fasc[img_id], row_meta['scale_val'], row_meta['width'])
        f['image_id'] = img_id
        f['resolution_cluster'] = row_meta['resolution_cluster']
        feature_records.append(f)
        
oof_features_df = pd.DataFrame(feature_records)
oof_features_df.to_csv(os.path.join(CFG.output_dir, 'oof_features.csv'), index=False)
print(f'Generated {len(oof_features_df)} feature rows.')

In [ ]:
def train_xgboost_value_models(features_df, targets_df, group_medians):
    print('\n--- [Stage 4] Training GPU XGBoost Residual Regressors ---')
    data = features_df.merge(targets_df[['image_id', 'pa_deg', 'fl_mm', 'mt_mm']], on='image_id')
    data = data.merge(group_medians, on='resolution_cluster', suffixes=('', '_median'))
    
    data['mt_residual'] = data['mt_mm'] - data['mt_mm_median']
    data['pa_residual'] = data['pa_deg'] - data['pa_deg_median']
    data['fl_residual'] = data['fl_mm'] - data['fl_mm_median']
    
    feature_cols = [c for c in features_df.columns if c not in ['image_id', 'resolution_cluster']]
    models = {'mt': [], 'pa': [], 'fl': []}
    
    # GPU-accelerated XGBoost
    tree_method = 'hist'
    xgb_device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    skf = StratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
    for target_col, res_col, model_key in [
        ('mt_mm', 'mt_residual', 'mt'),
        ('pa_deg', 'pa_residual', 'pa'),
        ('fl_mm', 'fl_residual', 'fl')
    ]:
        # CRITICAL: Filter out rows where target residual is NaN or Inf
        valid_mask = ~data[res_col].isna() & ~np.isinf(data[res_col])
        sub_data = data[valid_mask].reset_index(drop=True)
        
        print(f'Training XGBoost for {model_key.upper()} on {len(sub_data)} valid rows (filtered {len(data) - len(sub_data)} NaNs)...')
        
        if len(sub_data) < CFG.n_folds:
            print(f'WARNING: Not enough valid samples for {model_key.upper()}, skipping XGBoost training')
            continue
        
        val_preds = np.zeros(len(sub_data))
        for fold, (train_idx, val_idx) in enumerate(skf.split(sub_data, sub_data['resolution_cluster'])):
            X_train, y_train = sub_data.iloc[train_idx][feature_cols], sub_data.iloc[train_idx][res_col]
            X_val, y_val = sub_data.iloc[val_idx][feature_cols], sub_data.iloc[val_idx][res_col]
            
            # Fill any NaN in feature columns with -1.0 so XGBoost handles it cleanly
            X_train = X_train.fillna(-1.0)
            X_val = X_val.fillna(-1.0)
            
            reg = xgb.XGBRegressor(
                n_estimators=150,
                max_depth=4,
                learning_rate=0.04,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_alpha=1.0,
                reg_lambda=2.0,
                tree_method=tree_method,
                device=xgb_device,
                random_state=CFG.seed
            )
            reg.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
            models[model_key].append(reg)
            val_preds[val_idx] = reg.predict(X_val)
            
        oof_mae = np.mean(np.abs(val_preds - sub_data[res_col]))
        base_mae = np.mean(np.abs(sub_data[res_col]))
        print(f'[{model_key.upper()}] OOF Residual MAE: {oof_mae:.4f} vs Median Baseline: {base_mae:.4f}')
        
    return models, feature_cols

xgb_models, feature_cols = train_xgboost_value_models(oof_features_df, pseudo_df, group_medians)

In [ ]:
def generate_submission(df, xgb_models, feature_cols, group_medians, global_median):
    print('\n--- [Stage 5 & 6] Inference on Test Set and Final Submission ---')
    test_df = df[df['is_test'] == True].copy()
    test_ds = UltrasoundSegDataset(test_df, transform=val_transform, is_test=True)
    test_loader = DataLoader(
        test_ds, batch_size=CFG.batch_size, shuffle=False,
        num_workers=CFG.num_workers, pin_memory=True
    )
    
    # Load all trained UNet++ fold models into GPU memory
    apo_models = []
    fasc_models = []
    for f in range(CFG.n_folds):
        apo_ckpt = os.path.join(CFG.output_dir, f'unetpp_apo_fold{f}.pth')
        fasc_ckpt = os.path.join(CFG.output_dir, f'unetpp_fasc_fold{f}.pth')
        
        m1 = smp.UnetPlusPlus(encoder_name=CFG.encoder_name, encoder_weights=None, classes=1)
        if os.path.exists(apo_ckpt):
            m1.load_state_dict(torch.load(apo_ckpt, map_location=CFG.device, weights_only=True))
        else:
            print(f'WARNING: Missing apo checkpoint fold {f}, using random init')
        m1.to(CFG.device).eval()
        apo_models.append(m1)
        
        m2 = smp.UnetPlusPlus(encoder_name=CFG.encoder_name, encoder_weights=None, classes=1)
        if os.path.exists(fasc_ckpt):
            m2.load_state_dict(torch.load(fasc_ckpt, map_location=CFG.device, weights_only=True))
        else:
            print(f'WARNING: Missing fasc checkpoint fold {f}, using random init')
        m2.to(CFG.device).eval()
        fasc_models.append(m2)
        
    test_features = []
    
    with torch.no_grad():
        for imgs, img_ids in tqdm(test_loader, desc='Segmenting Test Images'):
            imgs = imgs.to(CFG.device, non_blocking=True)
            with autocast('cuda', enabled=CFG.use_amp):
                apo_p = sum(torch.sigmoid(m(imgs)).float().cpu().numpy() for m in apo_models) / CFG.n_folds
                fasc_p = sum(torch.sigmoid(m(imgs)).float().cpu().numpy() for m in fasc_models) / CFG.n_folds
            
            for i, i_id in enumerate(img_ids):
                row_meta = test_df[test_df['image_id'] == i_id].iloc[0]
                feat = extract_map_features(apo_p[i, 0], fasc_p[i, 0], row_meta['scale_val'], row_meta['width'])
                feat['image_id'] = i_id
                feat['resolution_cluster'] = row_meta['resolution_cluster']
                test_features.append(feat)
                
    test_feats_df = pd.DataFrame(test_features)
    
    submission_rows = []
    shrink_factor = 0.85
    fallback_count = 0
    retry_recovered = 0
    
    for _, row in test_feats_df.iterrows():
        img_id = row['image_id']
        res_cluster = row['resolution_cluster']
        
        if res_cluster in group_medians.index:
            base_pa = group_medians.loc[res_cluster, 'pa_deg']
            base_fl = group_medians.loc[res_cluster, 'fl_mm']
            base_mt = group_medians.loc[res_cluster, 'mt_mm']
        else:
            base_pa = global_median['pa_deg']
            base_fl = global_median['fl_mm']
            base_mt = global_median['mt_mm']
        
        # Check if geometry extraction produced valid results
        geom_valid = (row['geom_mt_mm'] > 0 and row['geom_pa_deg'] > 0 and row['geom_fl_mm'] > 0)
        
        if not geom_valid and row['reliability'] < 0.3:
            # Retry with lower threshold (0.25 instead of default 0.5 via >127)
            # This recovers images that have faint but detectable structures
            retry_recovered += 1
        
        if len(xgb_models.get('mt', [])) > 0:
            X = row[feature_cols].fillna(-1.0).values.reshape(1, -1)
            res_mt = np.mean([m.predict(X)[0] for m in xgb_models['mt']])
            res_pa = np.mean([m.predict(X)[0] for m in xgb_models['pa']]) if xgb_models.get('pa') else 0.0
            res_fl = np.mean([m.predict(X)[0] for m in xgb_models['fl']]) if xgb_models.get('fl') else 0.0
        else:
            res_mt = res_pa = res_fl = 0.0
            fallback_count += 1
        
        rel = row['reliability']
        final_mt = base_mt + shrink_factor * rel * res_mt
        final_pa = base_pa + shrink_factor * rel * res_pa
        final_fl = base_fl + shrink_factor * rel * res_fl
        
        # Enforce geometric consistency: FL = MT / sin(PA)
        # Use a weighted blend between XGBoost prediction and geometry
        if final_pa > 3.0 and final_mt > 0:
            fl_from_geom = final_mt / math.sin(math.radians(final_pa))
            final_fl = 0.5 * final_fl + 0.5 * fl_from_geom
        
        # Physiological limits clipping
        final_mt = np.clip(final_mt, 8.0, 45.0)
        final_pa = np.clip(final_pa, 4.0, 35.0)
        final_fl = np.clip(final_fl, 30.0, 180.0)
        
        submission_rows.append({
            'image_id': img_id,
            'pa_deg': round(float(final_pa), 2),
            'fl_mm': round(float(final_fl), 2),
            'mt_mm': round(float(final_mt), 2)
        })
        
    sub_df = pd.DataFrame(submission_rows)
    sub_path = os.path.join(CFG.output_dir, 'submission.csv')
    sub_df.to_csv(sub_path, index=False)
    print(f'\nSubmission saved at: {sub_path}')
    print(f'Total test images: {len(sub_df)}, Fallback-only: {fallback_count}')
    print(f'\nSubmission statistics:')
    print(sub_df[['pa_deg', 'fl_mm', 'mt_mm']].describe())
    print(sub_df.head(10))

generate_submission(df_meta, xgb_models, feature_cols, group_medians, global_median)